# LIFE on Google Colab — PolitiFact++ (binary MF-vs-MR, LLaMA2-7B)

Faithful reproduction of the paper's setup: binary fake/real over the **LLM pair** (MF=fake, MR=real), with **LLaMA2-7B** as the reconstruction model. End-to-end: **convert → key-sentence extraction → concatenate → features → train**.

**Before you start:**
1. Set the Colab runtime to **GPU** — an **A100** is needed for LLaMA2-7B (Runtime → Change runtime type).
2. Upload the whole `LIFE` repo (including `dataset/data/`) to your Google Drive, e.g. `MyDrive/LIFE`.
3. Edit `PROJECT_DIR` in the path cell below if you put it somewhere else.
4. Step 3 uses the **ungated** `NousResearch/Llama-2-7b-hf` mirror by default — no HF token needed. (The official `meta-llama/Llama-2-7b-hf` is gated and requires an approved access request + token.)

Scope: **PolitiFact++** only (~229 LLM-pair articles: 97 fake + 132 real). VLPFN is excluded (its text has no punctuation, so sentence splitting cannot work). GossipCop++ is far heavier; try it only after this works.

In [37]:
# Confirm a GPU is attached
!nvidia-smi

Thu Jun 18 02:40:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [38]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
import os

# <-- change this if you uploaded the repo elsewhere
PROJECT_DIR = '/content/drive/MyDrive/LIFE'
os.chdir(PROJECT_DIR)

POLITIFACT_DIR = f'{PROJECT_DIR}/dataset/data/Fakenews-dataset-main/Fakenews-dataset-main/Dataset/PolitiFact++'

# Paper's binary task: LLM pair only (MF=fake, MR=real), reconstructed with LLaMA2-7B.
OUTPUT_BIN     = f'{PROJECT_DIR}/dataset/output_bin'        # MF_fake.jsonl + MR_true.jsonl
KEY_SENT       = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_top10.jsonl'
BERT_CKPT      = f'{PROJECT_DIR}/dataset/bert_bin.pt'       # fresh extractor for MF-vs-MR
FEATURES_LLAMA = f'{PROJECT_DIR}/dataset/features_llama'
TRAIN_PATH     = f'{PROJECT_DIR}/dataset/train_bin.jsonl'
TEST_PATH      = f'{PROJECT_DIR}/dataset/test_bin.jsonl'

print('cwd:', os.getcwd())
print('PolitiFact++ found:', os.path.isdir(POLITIFACT_DIR))

cwd: /content/drive/MyDrive/LIFE
PolitiFact++ found: True


In [40]:
# Install dependencies.
# If the fastNLP import fails at the training step, pin a compatible version:
#   !pip install -q fastNLP==1.0.1
!pip install -q -r requirements.txt

In [41]:
# NLTK sentence tokenizer data. 'punkt' gives english.pickle (used by train.py);
# 'punkt_tab' is required by newer nltk's sent_tokenize (used by step 1).
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## HuggingFace login (optional)
Step 3 defaults to the **ungated** `NousResearch/Llama-2-7b-hf` mirror, so **no token is needed — you can skip this cell**. Only run it if you switch Step 3 to the official gated `meta-llama/Llama-2-7b-hf` (which also requires an approved access request).

In [ ]:
# LLaMA-2 is gated on HuggingFace. First accept the license at
# https://huggingface.co/meta-llama/Llama-2-7b-hf, then run this cell and paste an
# access token from https://huggingface.co/settings/tokens
# (or replace with: login(token="hf_xxx")).
# from huggingface_hub import login
# login()

## Step 0 — Convert PolitiFact++ to the binary LLM-pair JSONL
`--subset llm` emits only `MF_fake.jsonl` (97, fake) and `MR_true.jsonl` (132, real) — the paper's binary task. HF/HR (human-written) are not used.

In [ ]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_BIN}" --subset llm

## Step 1 — Key-sentence extraction (top-10)
Trains a **fresh** BERT fake/real classifier on MF-vs-MR (saved to `BERT_CKPT`), then keeps the **top-10** most impactful sentences per article (paper's k for PolitiFact++). This is the slowest step (a forward pass per sentence per article).

In [ ]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_BIN}" --output_file "{KEY_SENT}" --top_k 10 --model_path "{BERT_CKPT}" --gpu 0

## Step 2 — Concatenate key sentences back into the data
Adds a `sentence` field to each record in `OUTPUT_BIN` by matching on `(id, label)`. **Overwrites the files in `OUTPUT_BIN` in place** — re-run Step 0 first if you need to reset them.

In [ ]:
!python dataset/2_concate.py --folder_path "{OUTPUT_BIN}" --important_sentences_file "{KEY_SENT}"

## Step 3 — Reconstruction probabilities with LLaMA2-7B
The paper's reconstruction model. A malicious prompt is prepended and LLaMA2-7B's per-token log-likelihoods over the key fragments form the "linguistic fingerprint" features. Loads in bfloat16 (~14 GB; needs the A100) and downloads ~13 GB on first run. Writes one feature JSONL per input file into `FEATURES_LLAMA`.

In [ ]:
# meta-llama/Llama-2-7b-hf is gated (needs Meta approval). NousResearch/Llama-2-7b-hf is an
# ungated mirror of the SAME weights/tokenizer — no token needed. Swap back to the official
# repo if/when your access request is approved.
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_BIN}" --output_dir "{FEATURES_LLAMA}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

## Step 4A — train the classifier (released head: BMES tags + CRF + majority vote)
Splits `FEATURES_LLAMA` into train/test (seed-0, deterministic) and trains the released Transformer classifier for **50 epochs** on the binary MF-vs-MR task. This head diverges from the paper — it tags tokens with B/M/E/S labels, CRF-decodes them, and recovers the article label by majority vote — so it is the **A-side** of the head A/B. Step 4B below is the paper-faithful head. Paper target for PolitiFact++: **Acc 0.900 / F1 0.882**.

In [ ]:
!python LIFE_train/train.py \
  --split_dataset \
  --data_path "{FEATURES_LLAMA}" \
  --train_path "{TRAIN_PATH}" \
  --test_path "{TEST_PATH}" \
  --model Transformer \
  --num_train_epochs 50

## Step 4B — paper head (sigmoid + BCE, Eq 11–12)
Same CNN→Transformer trunk, but the head matches the paper: masked mean-pool → one sigmoid probability per article, trained with **binary cross-entropy** (fake=1, real=0) and evaluated directly at article level — no BMES tags, no CRF, no majority vote. Files: `LIFE_train/model_bce.py` + `LIFE_train/train_bce.py` (the originals are untouched and remain the A-side).

**The A/B is fair:** 4A and 4B consume the *same* `FEATURES_LLAMA` and the *same* seed-0 train/test split (`--split_dataset` here regenerates the identical split, so running 4A first is not required). The test set is only ~46 articles (≈2 accuracy points per article), so **re-run this cell with `--seed 1`, `2`, `3`, `4` and report mean ± std**. A-side references: 85.5/80.8 and 83.9/78.2; paper target 90.0/88.2. Checkpoint: `bce_en.pt`.

In [ ]:
#!python LIFE_train/train_bce.py \
#  --split_dataset \
#  --data_path "{FEATURES_LLAMA}" \
#  --train_path "{TRAIN_PATH}" \
#  --test_path "{TEST_PATH}" \
#  --num_train_epochs 50 \
#  --seed 0

## Multiclass experiment — 4-class HF / HR / MF / MR (LLaMA2-7B, released CRF/BMES head)

A separate, exploratory run that classifies all **four** PolitiFact++ categories
(human-fake, human-true, gpt3.5-fake, gpt3.5-true) instead of the paper's binary MF-vs-MR.
It reuses the released CRF/BMES head via `LIFE_train/train_multi.py` (a copy of `train.py`
with `en_labels` set to the four classes; `model.py` / `dataloader.py` are imported unchanged).

This needs its own LLaMA2-7B features (the binary `FEATURES_LLAMA` only has MF/MR), so
Steps 0m–3m re-run the pipeline with `--subset all` into separate `*_multi` paths — nothing
above is overwritten. Reference: an earlier 4-class run with **gpt2** features scored ~51.7%;
this swaps in the LLaMA2-7B features.

In [42]:
# --- 4-class (HF/HR/MF/MR) experiment paths (separate from the binary run above) ---
OUTPUT_MULTI     = f'{PROJECT_DIR}/dataset/output_multi'
KEY_SENT_MULTI   = f'{PROJECT_DIR}/dataset/keySentence/important_sentences_multi_top10.jsonl'
BERT_CKPT_MULTI  = f'{PROJECT_DIR}/dataset/bert_multi.pt'
FEATURES_MULTI   = f'{PROJECT_DIR}/dataset/features_llama_multi'
TRAIN_PATH_MULTI = f'{PROJECT_DIR}/dataset/train_multi.jsonl'
TEST_PATH_MULTI  = f'{PROJECT_DIR}/dataset/test_multi.jsonl'

### Step 0m — Convert all four categories
`--subset all` emits `HF_fake` / `MF_fake` / `HR_true` / `MR_true` JSONL (~520 articles).

In [43]:
!python dataset/0_convert.py --input_dir "{POLITIFACT_DIR}" --output_dir "{OUTPUT_MULTI}" --subset all

HF.json -> HF_fake.jsonl: 97 records (label=human_fake)
MF.json -> MF_fake.jsonl: 97 records (label=gpt3.5_fake)
HR.json -> HR_true.jsonl: 194 records (label=human_true)
MR.json -> MR_true.jsonl: 132 records (label=gpt3.5_true)


### Step 1m — Key-sentence extraction (top-10)
Trains a fresh binary BERT (fake = HF+MF, true = HR+MR) and keeps the top-10 sentences per
article. Slowest step; now over ~520 articles.

In [44]:
!python dataset/1_keySentenceExtraction.py --data_dir "{OUTPUT_MULTI}" --output_file "{KEY_SENT_MULTI}" --top_k 10 --model_path "{BERT_CKPT_MULTI}" --gpu 0

Loading weights: 100% 199/199 [00:00<00:00, 4996.39it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider

### Step 2m — Concatenate key sentences
Adds the `sentence` field to the records in `OUTPUT_MULTI` **in place** — re-run Step 0m to reset.

In [45]:
!python dataset/2_concate.py --folder_path "{OUTPUT_MULTI}" --important_sentences_file "{KEY_SENT_MULTI}"

所有 .jsonl 文件已成功更新。


### Step 3m — LLaMA2-7B reconstruction features
Same as the binary Step 3 but over all four files → `FEATURES_MULTI`.

In [46]:
!python dataset/3_gen_features_local.py --input_dir "{OUTPUT_MULTI}" --output_dir "{FEATURES_MULTI}" --model NousResearch/Llama-2-7b-hf --scorer llama --dtype bfloat16 --gpu 0

Streaming output truncated to the last 5000 lines.
58 80
82 112
114 410
413 427
  7% 7/97 [00:00<00:09,  9.64it/s]0 117
118 202
204 261
319 349
444 549
551 585
600 622
624 652
654 692
763 808
810 827
0 57
58 108
  9% 9/97 [00:01<00:07, 11.20it/s]0 117
182 182
233 233
235 285
739 739
769 817
820 820
822 850
852 872
874 902
905 905
0 117
506 566
806 848
850 868
870 893
927 957
959 1181
1183 1309
1312 1330
1332 1455
1457 1481
 11% 11/97 [00:01<00:08, 10.41it/s]0 57
58 136
0 117
118 249
359 473
504 518
521 682
751 791
857 960
1117 1165
1168 1226
1229 1261
1264 1270
 13% 13/97 [00:01<00:07, 10.82it/s]0 57
58 90
92 126
129 159
161 191
194 208
0 57
58 86
0 117
389 441
473 483
692 736
940 980
982 1024
1027 1054
1056 1082
1084 1106
1108 1130
1132 1150
 16% 16/97 [00:01<00:06, 12.19it/s]0 57
58 158
160 200
202 234
236 266
268 314
0 117
158 180
182 220
223 293
295 367
378 458
461 505
507 571
573 605
614 654
656 710
 19% 18/97 [00:01<00:05, 13.23it/s]0 117
118 164
166 246
248 387
389 455
457 525
6

### Step 4m — Train the 4-class classifier
Released CRF/BMES head over 16 BMES tags (4 classes × B/M/E/S), recovered to a 4-class label
by sentence majority vote. Prints Accuracy / Macro-F1 / per-class precision-recall in the
class id order printed at startup. Writes `linear_multi_en.pt`.

In [47]:
!python LIFE_train/train_multi.py --split_dataset --data_path "{FEATURES_MULTI}" --train_path "{TRAIN_PATH_MULTI}" --test_path "{TEST_PATH_MULTI}" --model Transformer --num_train_epochs 50

Log INFO: split dataset...
********************************
The overall data sources:
['HF_fake.jsonl', 'HR_true.jsonl', 'MF_fake.jsonl', 'MR_true.jsonl']
100% 416/416 [00:00<00:00, 2852.68it/s]
100% 104/104 [00:00<00:00, 3159.00it/s]

The number of train dataset: 416
The number of test  dataset: 104
********************************
classes (id order): {'human_fake': 0, 'human_true': 1, 'gpt3.5_fake': 2, 'gpt3.5_true': 3}
100% 416/416 [00:00<00:00, 1533.97it/s]
100% 104/104 [00:00<00:00, 440.59it/s]
--------------------------------classify--------------------------------
Log INFO: do train...
Epoch:   0% 0/50 [00:00<?, ?it/s]
Iteration:   0% 0/13 [00:00<?, ?it/s]
Iteration:   8% 1/13 [00:00<00:07,  1.61it/s]
Iteration:  15% 2/13 [00:00<00:03,  2.87it/s]
Iteration:  23% 3/13 [00:00<00:02,  4.01it/s]
Iteration:  31% 4/13 [00:01<00:01,  4.88it/s]
Iteration:  38% 5/13 [00:01<00:01,  5.55it/s]
Iteration:  46% 6/13 [00:01<00:01,  5.93it/s]
Iteration:  54% 7/13 [00:01<00:00,  6.10it/s]
Iterat

## Notes / troubleshooting
- **HF gating**: Step 3 defaults to the ungated `NousResearch/Llama-2-7b-hf` mirror (no token). If you switch to the official `meta-llama` repo and hit a 403 "gated repo", your access request hasn't been approved yet.
- **fastNLP**: if Step 4 errors on `from fastNLP.modules.torch import ...`, run `!pip install -q fastNLP==1.0.1` and restart the runtime.
- **Checkpoints**: `BERT_CKPT` (step 1) and `linear_en.pt` (step 4) are written under `PROJECT_DIR` on Drive, so they survive disconnects.
- **NaN features**: if Step 3 prints NaN/inf, switch Step 3 to `--dtype float32` (fits the 40 GB A100).
- **Re-runs**: Step 2 mutates `OUTPUT_BIN` in place; always re-run Step 0 before re-running Steps 1–3 from scratch.